In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE

In [19]:
df = pd.read_csv('../data/processed/feature_engineered_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

In [20]:
df_1 = pd.read_csv('../data/processed/cleaned_data.csv')

In [21]:
df_1['Date'] = pd.to_datetime(df_1['Date'])

In [22]:
df = df.merge(df_1[['Date', 'Close']], on='Date', how='left')

In [23]:
df

,Date,Target,High,Low,Volume,SP_Ajclose,SP_volume,DJ_low,DJ_volume,EG_volume,...,PLT_Price,PLT_Open,PLD_Price,USDI_Open,USDI_Volume,GDX_Close,GDX_Adj Close,GDX_Volume,USO_Volume,Close
0,2011-12-15,155.229996,154.949997,151.710007,21521900,105.441238,199109200,11825.21973,136930000,787900,...,1414.65,1420.30,618.85,80.565,22850,51.680000,48.973877,20605600,12616700,152.330002
1,2011-12-16,154.869995,155.369995,153.899994,18124300,105.597549,220481400,11819.30957,389520000,896600,...,1420.25,1414.75,623.65,80.175,13150,52.680000,49.921513,16285400,12578800,155.229996
2,2011-12-19,156.979996,155.860001,154.360001,12547200,104.468536,183903000,11735.19043,135170000,2096700,...,1411.10,1422.65,608.80,80.300,970,51.169998,48.490578,15120200,7418200,154.869995
3,2011-12-20,157.160004,157.429993,156.580002,9136300,107.629784,225418100,11768.83008,165180000,875300,...,1434.75,1408.95,626.65,80.890,22950,52.990002,50.215282,11644900,10041600,156.979996
4,2011-12-21,156.039993,157.529999,156.130005,11996100,107.838242,194230900,11999.44043,163250000,837600,...,1429.05,1434.40,635.90,80.105,24140,52.959999,50.186852,8724300,10728000,157.160004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1712,2018-12-21,120.019997,119.250000,118.570000,13629800,240.699997,255345600,22396.33984,900510000,2328500,...,789.05,796.50,1158.90,95.875,20130,20.440001,20.440001,63178700,38901200,118.720001
1713,2018-12-24,119.660004,120.139999,119.570000,9736400,234.339996,147311600,21792.19922,308420000,655100,...,788.40,791.85,1172.80,96.440,13930,21.090000,21.090000,60507000,21598200,120.019997
1714,2018-12-26,120.570000,121.000000,119.570000,14293500,246.179993,218485400,21712.52930,433080000,746300,...,799.25,788.75,1190.10,96.100,15660,20.620001,20.620001,76365200,40978800,119.660004
1715,2018-12-27,121.059998,120.900002,120.139999,11874400,248.070007,186267300,22267.41992,407940000,744000,...,795.50,799.40,1196.00,96.460,20520,20.969999,20.969999,52393000,36578700,120.570000


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1717 entries, 0 to 1716
Data columns (total 28 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           1717 non-null   datetime64[ns]
 1   Target         1717 non-null   float64       
 2   High           1717 non-null   float64       
 3   Low            1717 non-null   float64       
 4   Volume         1717 non-null   int64         
 5   SP_Ajclose     1717 non-null   float64       
 6   SP_volume      1717 non-null   int64         
 7   DJ_low         1717 non-null   float64       
 8   DJ_volume      1717 non-null   int64         
 9   EG_volume      1717 non-null   int64         
 10  EU_Price       1717 non-null   float64       
 11  OF_Volume      1717 non-null   int64         
 12  OS_Open        1717 non-null   float64       
 13  OS_High        1717 non-null   float64       
 14  SF_Low         1717 non-null   int64         
 15  SF_Volume      1717 n

In [25]:
df_stationary = df.copy()
price_cols = [c for c in df.columns if c != 'Date']

for col in price_cols:
    df_stationary[col] = df_stationary[col].pct_change()

df_stationary.dropna(inplace = True)

In [26]:
df_features = df.copy()

# 1. Exponential Moving Average (EMA)
df_features['EMA_12'] = df_features['Close'].ewm(span = 12, adjust = False).mean()
df_features['EMA_26'] = df_features['Close'].ewm(span = 26, adjust = False).mean()

# 2. Relative Strength Index (RSI)
delta = df_features['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window = 14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window = 14).mean()
rs = gain / (loss + 1e-10)
df_features['RSI'] = 100 - (100 / (1 + rs))

# 3. Daily Volatility (Standard Deviation)
df_features['Volatility_7'] = df_features['Close'].rolling(window = 7).std()

df_features.dropna(inplace = True)

In [27]:
# MACD Calculation
exp1 = df_features['Close'].ewm(span = 12, adjust = False).mean()
exp2 = df_features['Close'].ewm(span = 26, adjust = False).mean()
raw_macd = exp1 - exp2
raw_signal = raw_macd.ewm(span=9, adjust = False).mean()

# Divide by Close
df_features['MACD'] = raw_macd / df_features['Close']
df_features['Signal_Line'] = raw_signal / df_features['Close']

# Bollinger Bands
rolling_mean = df_features['Close'].rolling(window = 20).mean()
rolling_std = df_features['Close'].rolling(window = 20).std()
upper_band = rolling_mean + (rolling_std * 2)
lower_band = rolling_mean - (rolling_std * 2)
df_features['BB_Width'] = (upper_band - lower_band) / rolling_mean

df_features.dropna(inplace = True)

In [28]:
train_size = int(0.8 * len(df))

train_df = df.iloc[:train_size].copy()
test_df  = df.iloc[train_size:].copy()

In [29]:
date_train = train_df['Date'].reset_index(drop = True)
date_test  = test_df['Date'].reset_index(drop = True)

In [30]:
feature_cols = [c for c in df.columns if c not in ['Date', 'Target']]

In [31]:
x_scaler = MinMaxScaler(feature_range=(-1, 1))
y_scaler = MinMaxScaler(feature_range=(-1, 1))

In [32]:
X_train_scaled = x_scaler.fit_transform(train_df[feature_cols])
X_test_scaled  = x_scaler.transform(test_df[feature_cols])

y_train_scaled = y_scaler.fit_transform(train_df[['Target']].values)
y_test_scaled  = y_scaler.transform(test_df[['Target']].values)

In [33]:
train_df[feature_cols] = X_train_scaled
train_df['Target'] = y_train_scaled

test_df[feature_cols] = X_test_scaled
test_df['Target'] = y_test_scaled

In [34]:
train_df.to_csv('../data/processed/minmax_train_scaled.csv', index = False)
test_df.to_csv('../data/processed/minmax_test_scaled.csv', index = False)

joblib.dump(x_scaler, '../models/minmax_x_scaler.pkl')
joblib.dump(y_scaler, '../models/minmax_y_scaler.pkl')

print(f"Train: {train_df.shape} | Test: {test_df.shape}")

Train: (1373, 28) | Test: (344, 28)


In [35]:
train = pd.read_csv('../data/processed/minmax_train_scaled.csv')
test  = pd.read_csv('../data/processed/minmax_test_scaled.csv')

In [36]:
SEQ_LEN = 22

In [37]:
def make_sequences(df, seq_len):
    X = df[feature_cols].values
    y = df['Target'].values
    Xs, ys = [], []
    for i in range(len(df) - seq_len):
        Xs.append(X[i:(i + seq_len)])
        ys.append(y[i + seq_len])
    return np.array(Xs), np.array(ys)

In [38]:
X_train, y_train = make_sequences(train, SEQ_LEN)
X_test,  y_test  = make_sequences(test, SEQ_LEN)

In [39]:
np.savez(
    '../data/processed/cnn_lstm_ready.npz',
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test,
    feature_names = feature_cols,
    seq_len = SEQ_LEN
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (1351, 22, 26) | Test: (322, 22, 26)
